# Розширення генерації на основі пошуку (RAG) та векторні бази даних

In [1]:
%pip install getenv openai faiss-cpu pandas numpy


[notice] A new release of pip is available: 25.1.1 -> 25.3
[notice] To update, run: python -m pip install --upgrade pip
Note: you may need to restart the kernel to use updated packages.


In [42]:
import os
import pandas as pd
import numpy as np
import faiss

from dotenv import load_dotenv

load_dotenv()

True

## Створення нашої бази знань

Налаштування FAISS для векторного пошуку


In [ ]:
# Шляхи до файлів (даних для RAG)
data_paths = [
    "data/frameworks.md", 
    "data/own_framework.md", 
    "data/perceptron.md"
] 

# Ініціалізація порожнього DataFrame
df = pd.DataFrame(columns=['path', 'text'])

# Сучасний спосіб додавання рядків до DataFrame
for path in data_paths:
    try:
        with open(path, 'r', encoding='utf-8') as file:
            file_content = file.read()
        
        # Використовуємо concat замість застарілого append
        new_row = pd.DataFrame({'path': [path], 'text': [file_content]})
        df = pd.concat([df, new_row], ignore_index=True)
    except FileNotFoundError:
        print(f"Файл не знайдено: {path}")

df.head()

In [ ]:
def split_text(text, max_length, min_length):
    words = text.split()
    chunks = []
    current_chunk = []

    for word in words:
        current_chunk.append(word)
        if len(' '.join(current_chunk)) < max_length and len(' '.join(current_chunk)) > min_length:
            chunks.append(' '.join(current_chunk))
            current_chunk = []

    # Якщо останній фрагмент не досягнув мінімальної довжини, все одно додати його
    if current_chunk:
        chunks.append(' '.join(current_chunk))

    return chunks

# Припускаючи, що analyzed_df - це pandas DataFrame, а 'output_content' - це стовпець у цьому DataFrame
splitted_df = df.copy()
splitted_df['chunks'] = splitted_df['text'].apply(lambda x: split_text(x, 400, 300))

splitted_df

In [ ]:
# Припускаючи, що 'chunks' - це стовпець списків у DataFrame splitted_df, ми розділимо фрагменти на різні рядки
flattened_df = splitted_df.explode('chunks')

flattened_df.head()

## Перетворення тексту на ембедінги

In [70]:
from azure.ai.inference import EmbeddingsClient
from azure.core.credentials import AzureKeyCredential

endpoint = "https://models.inference.ai.azure.com"
token = os.environ["GITHUB_TOKEN"]

embed_model_name = "cohere-embed-v3-multilingual" 

embed_client = EmbeddingsClient(
        endpoint=endpoint,
        credential=AzureKeyCredential(token)
)

def create_embeddings(text):
    """
    Створює ембедінги для тексту.
    
    Args:
        text: Текст або список текстів для ембедінгу
        model: Модель для ембедінгу (за замовчуванням mistral-embed)
        
    Returns:
        Вектор ембедінгу
    """
    # Обробка pandas Series
    if isinstance(text, pd.Series):
        # Беремо перший елемент з Series
        text = text.iloc[0]
    
    # Перетворюємо в список рядків для API
    if not isinstance(text, list):
        text = [str(text)]
    else:
        text = [str(item) for item in text]
    
    embeddings_response = embed_client.embed(
        input=text,
        model=embed_model_name
    )
    
    # Повернення ембедінгу для першого елемента
    return embeddings_response.data[0].embedding

# Приклад використання:
embeddings = create_embeddings(flattened_df['chunks'][0])

[0.00630188,
 0.010032654,
 -0.0020694733,
 -0.015914917,
 -0.02796936,
 0.007827759,
 -0.005268097,
 -0.036468506,
 -0.005508423,
 0.0060577393,
 0.023132324,
 0.013717651,
 0.0005393028,
 0.010406494,
 0.005924225,
 0.009422302,
 0.03390503,
 -0.021362305,
 0.010467529,
 -0.011383057,
 -0.0012874603,
 0.018737793,
 0.042938232,
 0.0024642944,
 -0.036132812,
 0.043304443,
 0.016082764,
 -0.036895752,
 0.018218994,
 -0.029144287,
 -0.016052246,
 -0.0056266785,
 0.027389526,
 0.028289795,
 -0.022781372,
 0.021118164,
 -0.022994995,
 -0.049987793,
 0.0027751923,
 0.04925537,
 0.010757446,
 0.02684021,
 -0.020858765,
 0.023529053,
 -0.06488037,
 0.014717102,
 0.013954163,
 0.028549194,
 0.026428223,
 0.023391724,
 0.00060892105,
 0.002544403,
 0.023666382,
 -0.002866745,
 0.027145386,
 -0.00818634,
 0.002204895,
 0.0050354004,
 -0.0060272217,
 -0.0025501251,
 0.03152466,
 0.016815186,
 -0.034301758,
 0.04663086,
 -0.014587402,
 0.06756592,
 0.043914795,
 0.03741455,
 0.018051147,
 0.01948

# Розширення генерації на основі пошуку (RAG) та векторні бази даних

In [71]:
cat = create_embeddings("cat")

cat

[0.00630188,
 0.010032654,
 -0.0020694733,
 -0.015914917,
 -0.02796936,
 0.007827759,
 -0.005268097,
 -0.036468506,
 -0.005508423,
 0.0060577393,
 0.023132324,
 0.013717651,
 0.0005393028,
 0.010406494,
 0.005924225,
 0.009422302,
 0.03390503,
 -0.021362305,
 0.010467529,
 -0.011383057,
 -0.0012874603,
 0.018737793,
 0.042938232,
 0.0024642944,
 -0.036132812,
 0.043304443,
 0.016082764,
 -0.036895752,
 0.018218994,
 -0.029144287,
 -0.016052246,
 -0.0056266785,
 0.027389526,
 0.028289795,
 -0.022781372,
 0.021118164,
 -0.022994995,
 -0.049987793,
 0.0027751923,
 0.04925537,
 0.010757446,
 0.02684021,
 -0.020858765,
 0.023529053,
 -0.06488037,
 0.014717102,
 0.013954163,
 0.028549194,
 0.026428223,
 0.023391724,
 0.00060892105,
 0.002544403,
 0.023666382,
 -0.002866745,
 0.027145386,
 -0.00818634,
 0.002204895,
 0.0050354004,
 -0.0060272217,
 -0.0025501251,
 0.03152466,
 0.016815186,
 -0.034301758,
 0.04663086,
 -0.014587402,
 0.06756592,
 0.043914795,
 0.03741455,
 0.018051147,
 0.01948

In [72]:
import pickle
from pathlib import Path

def save_embeddings(df, folder="embeddings", filename="flattened_df.pkl"):
    """
    Зберігає DataFrame з ембедінгами в указану папку.
    
    Args:
        df: DataFrame з ембедінгами
        folder: Назва папки для збереження
        filename: Ім'я файлу для збереження
    """
    # Створення директорії, якщо вона не існує
    Path(folder).mkdir(parents=True, exist_ok=True)
    
    # Шлях до файлу
    file_path = os.path.join(folder, filename)
    
    # Збереження DataFrame
    with open(file_path, 'wb') as f:
        pickle.dump(df, f)
    
    print(f"DataFrame успішно збережено в {file_path}")

def load_embeddings(folder="embeddings", filename="flattened_df.pkl"):
    """
    Завантажує DataFrame з ембедінгами з указаної папки.
    
    Args:
        folder: Назва папки для завантаження
        filename: Ім'я файлу для завантаження
        
    Returns:
        DataFrame з ембедінгами або None, якщо файл не існує
    """
    # Шлях до файлу
    file_path = os.path.join(folder, filename)
    
    # Перевірка існування файлу
    if os.path.exists(file_path):
        # Завантаження DataFrame
        with open(file_path, 'rb') as f:
            df = pickle.load(f)
        
        print(f"DataFrame успішно завантажено з {file_path}")
        return df
    else:
        print(f"Файл {file_path} не знайдено")
        return None

def get_or_create_embeddings(df, chunk_column, embedding_function, folder="embeddings", filename="flattened_df.pkl"):
    """
    Завантажує DataFrame з ембедінгами або створює новий.
    
    Args:
        df: Вихідний DataFrame з текстами
        chunk_column: Назва стовпця з текстовими фрагментами
        embedding_function: Функція для створення ембедінгів
        folder: Назва папки для збереження/завантаження
        filename: Ім'я файлу для збереження/завантаження
        
    Returns:
        DataFrame з ембедінгами
    """
    # Спроба завантажити DataFrame
    loaded_df = load_embeddings(folder, filename)
    
    if loaded_df is not None:
        return loaded_df
    
    # Якщо завантаження не вдалося, створюємо ембедінги
    print("Створення нових ембедінгів...")
    
    # Створення ембедінгів
    embeddings = []
    for chunk in df[chunk_column]:
        embeddings.append(embedding_function(chunk))
    
    # Збереження ембедінгів в DataFrame
    df['embeddings'] = embeddings
    
    # Збереження DataFrame
    save_embeddings(df, folder, filename)
    
    return df


# Використовуємо функцію, яка розраховує ембедінги, 
# якщо вони не були раніше створені і збережені в папці в "15-rag-and-vector-databases/embeddings"
flattened_df = get_or_create_embeddings(
    splitted_df.explode('chunks'), 
    'chunks', 
    create_embeddings
)

flattened_df.head()

Файл embeddings/flattened_df.pkl не знайдено
Створення нових ембедінгів...
DataFrame успішно збережено в embeddings/flattened_df.pkl


,path,text,chunks,embeddings
0,data/frameworks.md,# Neural Network Frameworks\n\nAs we have lear...,# Neural Network Frameworks As we have learned...,"[0.005393982, 0.018859863, 0.0063819885, 0.009..."
0,data/frameworks.md,# Neural Network Frameworks\n\nAs we have lear...,descent optimization While the `numpy` library...,"[0.01537323, 0.0284729, -0.02949524, 0.0531921..."
0,data/frameworks.md,# Neural Network Frameworks\n\nAs we have lear...,should give us the opportunity to compute grad...,"[0.016677856, -0.004032135, -0.01838684, 0.051..."
0,data/frameworks.md,# Neural Network Frameworks\n\nAs we have lear...,those computations on GPUs is very important. ...,"[-0.0016927719, -0.01461792, 0.013427734, 0.03..."
0,data/frameworks.md,# Neural Network Frameworks\n\nAs we have lear...,"API, there is also higher-level API, called Ke...","[-0.018341064, -0.008888245, -0.017364502, 0.0..."


# Пошук з використанням FAISS

Векторний пошук та схожість між нашим запитом і базою даних з використанням FAISS

### Створення індексу FAISS та підготовка до пошуку

In [73]:
# Отримуємо ембедінги як масив numpy
embeddings_list = flattened_df['embeddings'].to_list()
embeddings_array = np.array(embeddings_list).astype('float32')

# Визначаємо розмірність векторів
vector_dimension = len(embeddings_list[0])

# Створюємо індекс FAISS
index = faiss.IndexFlatL2(vector_dimension)  # L2 - це евклідова відстань

# Додаємо наші вектори до індексу
index.add(embeddings_array)

# Перевіряємо кількість векторів в індексі
print(f"Загальна кількість векторів в індексі: {index.ntotal}, розмірність векторів: {vector_dimension}")

Загальна кількість векторів в індексі: 57, розмірність векторів: 1024


## Поєднання всього для відповіді на запитання

In [75]:
from azure.ai.inference import ChatCompletionsClient

client = ChatCompletionsClient(
    endpoint=endpoint,
    credential=AzureKeyCredential(token),
)

# Виберіть модель загального призначення для тексту
deployment = "gpt-4o-mini"

# Реалізація чатботів (при наявності і відсутності RAG)

In [13]:
def chatbot_with_rag(user_input):
    # Перетворіть запитання у вектор запиту
    query_vector = create_embeddings(user_input)
    query_vector_array = np.array([query_vector]).astype('float32')
    
    # Знайдіть найбільш схожі документи з FAISS
    k = 5  # кількість найближчих сусідів для пошуку
    distances, indices = index.search(query_vector_array, k)

    # додайте документи до запиту, щоб забезпечити контекст
    history = []
    for idx in indices[0]:
        history.append(flattened_df['chunks'].iloc[idx])

    # створюємо об'єкт повідомлення з контекстом
    context = "\n\n".join(history)  # всі знайдені фрагменти

    # Формуємо запит, що просить коротку, але завершену відповідь
    messages = [
        {"role": "system", "content": "You are an AI assistant that helps with AI questions. "},
        {"role": "user", "content": f"Context:\n{context}\n\nQuestion: {user_input}\n\n Provide a brief but complete answer based on the context. Answer in Ukrainian."}
    ]


    response = client.complete(
        temperature=0,
        model=deployment,
        messages=messages,
        max_tokens=300,
    )

    return response.choices[0].message.content


def chatbot_without_rag(user_input):
    """
    Чат-бот без використання RAG (прямий запит до моделі).
    """
    messages=[
        {"role": "system", "content": "You are an AI assistant that helps with AI questions. Provide brief but complete answers. Answer in Ukrainian."},
        {"role": "user", "content": f"Question: {user_input}"}
    ]

    response = client.complete(
        temperature=0,
        model=deployment,
        messages=messages,
        max_tokens=300,
    )

    return response.choices[0].message.content

# Порівняння відповідей RAG-системи

In [67]:
from IPython.display import display, Markdown, HTML

def compare_responses(user_input, save_to_file=False, filename="rag_comparison.md"):
    """
    Порівнює відповіді чат-боту з RAG та без RAG.
    
    Args:
        user_input: Запитання користувача
        save_to_file: Зберегти результат у файл Markdown
        filename: Назва файлу для збереження
    """
    # Отримання знайдених чанків
    query_vector = create_embeddings(user_input)

    query_vector_array = np.array([query_vector]).astype('float32')
    k = 5  # кількість найближчих сусідів для пошуку
    distances, indices = index.search(query_vector_array, k)
    
    # Отримання відповідей
    rag_response = chatbot_with_rag(user_input)
    no_rag_response = chatbot_without_rag(user_input)
    
    # Формування markdown-тексту
    markdown_text = f"""
# Порівняння відповідей

## 📝 Запит: {user_input}

## 🔍 Відповіді моделей

### Без використання RAG

{no_rag_response}

### З використанням RAG

{rag_response}

## 📚 Знайдені фрагменти тексту

"""
    
    # Додавання чанків
    for i, idx in enumerate(indices[0]):
        chunk_content = flattened_df['chunks'].iloc[idx]
        path = flattened_df['path'].iloc[idx]
        dist = float(distances[0][i])
        
        markdown_text += f"""
### Фрагмент {i+1} (відстань: {dist:.4f})

**Шлях**: {path}

{chunk_content}

"""
    
    # Виведення Markdown
    display(Markdown(markdown_text))
    
    # Для коректного відображення формул
    mathjax_script = """
    <script type="text/javascript">
        MathJax = {
            tex: {
                inlineMath: [['$', '$']]
            }
        };
    </script>
    <script type="text/javascript" id="MathJax-script" async
        src="https://cdn.jsdelivr.net/npm/mathjax@3/es5/tex-chtml.js">
    </script>
    """
    display(HTML(mathjax_script))
    
    # Збереження в файл, якщо потрібно
    if save_to_file:
        with open(filename, 'w', encoding='utf-8') as f:
            f.write(markdown_text)
        print(f"Результати збережено у файл: {filename}")


### Індивідуальне завдання

> Варіант 11 - Робототехніка

**Частина 1: Базова реалізація RAG-системи на власних даних**: Модифікуйте базовий ноутбук, щоб реалізувати RAG-систему для іншої тематики (згідно з вашим варіантом)

**Частина 2: Розширене завдання**: Розширте функціональність базового рішення, створивши інтерактивну систему для відповіді на запитання про навчальні курси. Така система матиме практичне застосування для навчальних закладів, освітніх платформ та онлайн-курсів

In [1]:
from azure.ai.inference import ChatCompletionsClient
from azure.core.credentials import AzureKeyCredential
from IPython.display import display, Markdown, clear_output
from dotenv import load_dotenv
import ipywidgets as widgets
import os

# Завантажуємо змінні середовища з .env
load_dotenv()

# Токен GitHub (має бути в .env як GITHUB_TOKEN=...)
token = os.getenv("GITHUB_TOKEN")
assert token, "GITHUB_TOKEN не заданий у .env або змінних середовища"

# Endpoint GitHub Models
endpoint = "https://models.inference.ai.azure.com"

# Клієнт для ChatCompletions
client = ChatCompletionsClient(
    endpoint=endpoint,
    credential=AzureKeyCredential(token),
)

deployment = "google/gemma-3n-e4b"


# ВІДПОВІДНІСТЬ файлів знань по робототехніці та «людських» назв курсів
ROBOTICS_COURSE_MAPPING = {
    "01_Robotics_Fundamentals.md": "Основи робототехніки",
    "02_Robot_Kinematics_and_Dynamics.md": "Кінематика та динаміка роботів",
    "03_Actuators_and_Sensors.md": "Актуатори та сенсори",
    "04_Robot_Control_Systems.md": "Системи керування роботами",
    "05_Mobile_Robots_and_SLAM.md": "Мобільні роботи та SLAM",
    "06_Industrial_Robots_and_Manipulators.md": "Промислові роботи та маніпулятори",
    "07_Human_Robot_Interaction.md": "Взаємодія людини та робота",
    "08_Robotics_AI_and_Perception.md": "ШІ та сприйняття в робототехніці",
    "09_Swarm_andMultiRobot_Systems.md": "Ройова та багатороботна робототехніка",
    "10_Robotics_Safety_and_Ethics.md": "Безпека та етика в робототехніці"
}


def get_robotics_course_name_from_path(path: str) -> str:
    """Отримує назву курсу з шляху до файлу (робототехніка)."""
    filename = path.split('/')[-1]
    return ROBOTICS_COURSE_MAPPING.get(filename, filename)


def find_robotics_course_by_topic(user_query):
    """
    Знаходить курси з робототехніки, що відповідають запиту користувача про певну тему.
    Використовує embeddings + FAISS-індекс (index, flattened_df).
    """
    query_vector = create_embeddings(user_query)
    query_vector_array = np.array([query_vector]).astype('float32')
    distances, indices = index.search(query_vector_array, 5)

    course_info = []
    course_names = set()

    for i, idx in enumerate(indices[0]):
        if idx < len(flattened_df):
            chunk = flattened_df["chunks"].iloc[idx]
            path = flattened_df["path"].iloc[idx]
            course_name = get_robotics_course_name_from_path(path)

            if course_name not in course_names:
                course_names.add(course_name)
                course_info.append(
                    {
                        "name": course_name,
                        "relevance": float(1.0 / (1.0 + distances[0][i])),
                        "context": chunk,
                        "file": path.split("/")[-1],
                    }
                )

    context = "\n\n".join(
        [f"Курс: {info['name']}\nОпис: {info['context']}" for info in course_info]
    )

    messages = [
        {
            "role": "system",
            "content": (
                "Ви освітній асистент з робототехніки, "
                "який допомагає знаходити відповідні курси навчання."
            ),
        },
        {
            "role": "user",
            "content": (
                f"На основі наступної інформації про курси:\n\n{context}\n\n"
                f"Порекомендуйте найбільш підходящі курси для запиту: '{user_query}'. "
                f"Зробіть структуровану відповідь з назвами курсів та коротким поясненням, "
                f"чому саме ці курси підходять."
            ),
        },
    ]

    response = client.complete(
        temperature=0.3,
        model=deployment,
        messages=messages,
        max_tokens=500,
    )

    return response.choices[0].message.content, course_info


def answer_robotics_course_question(course_name, user_question):
    """
    Відповідає на конкретні запитання про певний курс з робототехніки.
    """
    combined_query = f"Курс: {course_name}. Запитання: {user_question}"

    query_vector = create_embeddings(combined_query)
    query_vector_array = np.array([query_vector]).astype("float32")
    distances, indices = index.search(query_vector_array, 3)

    context_fragments = []
    sources = []
    for idx in indices[0]:
        if idx < len(flattened_df):
            context_fragments.append(flattened_df["chunks"].iloc[idx])
            sources.append(
                get_robotics_course_name_from_path(flattened_df["path"].iloc[idx])
            )

    context = "\n\n".join(context_fragments)

    messages = [
        {
            "role": "system",
            "content": (
                "Ви освітній асистент з робототехніки, "
                "який відповідає на запитання про навчальні курси."
            ),
        },
        {
            "role": "user",
            "content": (
                f"Інформація про курс '{course_name}':\n\n{context}\n\n"
                f"Дайте детальну відповідь на запитання: '{user_question}', "
                f"використовуючи тільки надану інформацію. Якщо інформації недостатньо, "
                f"чесно визнайте це."
            ),
        },
    ]

    response = client.complete(
        temperature=0.2,
        model=deployment,
        messages=messages,
        max_tokens=400,
    )

    return response.choices[0].message.content, sources


def chatbot_with_rag_robotics(user_input):
    """Чат-бот з використанням RAG для теми робототехніки."""
    query_vector = create_embeddings(user_input)
    query_vector_array = np.array([query_vector]).astype("float32")

    k = 5
    distances, indices = index.search(query_vector_array, k)

    history = []
    sources = []
    for idx in indices[0]:
        if idx < len(flattened_df):
            history.append(flattened_df["chunks"].iloc[idx])
            sources.append(
                get_robotics_course_name_from_path(flattened_df["path"].iloc[idx])
            )

    context = "\n\n".join(history)

    messages = [
        {
            "role": "system",
            "content": "Ви експерт з робототехніки. Відповідайте українською мовою.",
        },
        {
            "role": "user",
            "content": (
                f"Контекст:\n{context}\n\n"
                f"Запитання: {user_input}\n\n"
                f"Дайте коротку, але повну відповідь."
            ),
        },
    ]

    response = client.complete(
        temperature=0.3,
        model=deployment,
        messages=messages,
        max_tokens=400,
    )

    return response.choices[0].message.content, list(set(sources))


def chatbot_without_rag_robotics(user_input):
    """Чат-бот без використання RAG (тільки модель, тема – робототехніка)."""
    messages = [
        {
            "role": "system",
            "content": "Ви експерт з робототехніки. Відповідайте українською мовою.",
        },
        {"role": "user", "content": f"Запитання: {user_input}"},
    ]

    response = client.complete(
        temperature=0.3,
        model=deployment,
        messages=messages,
        max_tokens=400,
    )

    return response.choices[0].message.content


def educational_assistant_robotics():
    """
    Інтерактивний освітній асистент з робототехніки з меню вибору функцій.
    """
    # Віджети
    query_text = widgets.Textarea(
        value="",
        placeholder="Введіть ваш запит про робототехніку...",
        description="Запит:",
        disabled=False,
        layout=widgets.Layout(width="100%", height="80px"),
    )

    course_dropdown = widgets.Dropdown(
        options=[("-- Виберіть курс --", "")] + [
            (name, name) for name in ROBOTICS_COURSE_MAPPING.values()
        ],
        value="",
        description="Курс:",
        disabled=False,
        layout=widgets.Layout(width="100%"),
    )

    action_dropdown = widgets.Dropdown(
        options=[
            ("🔍 Знайти курси за темою", "find"),
            ("❓ Запитати про конкретний курс", "ask"),
            ("💬 Загальне питання з RAG", "rag"),
            ("🤖 Порівняти RAG vs без RAG", "compare"),
        ],
        value="find",
        description="Дія:",
        disabled=False,
        layout=widgets.Layout(width="100%"),
    )

    output = widgets.Output()

    def on_button_clicked(_):
        with output:
            clear_output()

            if action_dropdown.value == "find":
                if not query_text.value.strip():
                    display(
                        Markdown(
                            "❌ Будь ласка, введіть запит про бажану тему навчання."
                        )
                    )
                    return

                display(
                    Markdown(
                        f"## 🔍 Пошук курсів з робототехніки за темою: *{query_text.value}*\n---"
                    )
                )

                try:
                    response, courses = find_robotics_course_by_topic(query_text.value)

                    display(Markdown("### 📚 Знайдені курси:"))
                    for i, course in enumerate(courses[:5], 1):
                        relevance_percent = course["relevance"] * 100
                        display(
                            Markdown(
                                f"{i}. {course['name']} (релевантність: {relevance_percent:.1f}%)"
                            )
                        )

                    display(Markdown("---\n### 🤖 Рекомендація асистента:"))
                    display(Markdown(response))
                except Exception as e:
                    display(Markdown(f"❌ Помилка: {str(e)}"))

            elif action_dropdown.value == "ask":
                if not course_dropdown.value or not query_text.value.strip():
                    display(
                        Markdown(
                            "❌ Будь ласка, виберіть курс та введіть ваше запитання."
                        )
                    )
                    return

                display(
                    Markdown(
                        f"## ❓ Запитання про курс: *{course_dropdown.value}*"
                    )
                )
                display(
                    Markdown(f"### Запитання: *{query_text.value}*\n---")
                )

                try:
                    response, sources = answer_robotics_course_question(
                        course_dropdown.value, query_text.value
                    )
                    display(Markdown("### 📝 Відповідь:"))
                    display(Markdown(response))

                    if sources:
                        display(
                            Markdown(
                                f"---\nДжерела: {', '.join(set(sources))}"
                            )
                        )
                except Exception as e:
                    display(Markdown(f"❌ Помилка: {str(e)}"))

            elif action_dropdown.value == "rag":
                if not query_text.value.strip():
                    display(
                        Markdown("❌ Будь ласка, введіть ваше запитання.")
                    )
                    return

                display(
                    Markdown(
                        f"## 💬 Загальне питання: *{query_text.value}*\n---"
                    )
                )

                try:
                    response, sources = chatbot_with_rag_robotics(
                        query_text.value
                    )
                    display(
                        Markdown(
                            "### 📝 Відповідь (з використанням RAG):"
                        )
                    )
                    display(Markdown(response))

                    if sources:
                        display(
                            Markdown(
                                f"---\nВикористані джерела: {', '.join(sources)}"
                            )
                        )
                except Exception as e:
                    display(Markdown(f"❌ Помилка: {str(e)}"))

            elif action_dropdown.value == "compare":
                if not query_text.value.strip():
                    display(
                        Markdown("❌ Будь ласка, введіть ваше запитання.")
                    )
                    return

                display(
                    Markdown(
                        f"## 🔄 Порівняння відповідей для: *{query_text.value}*\n---"
                    )
                )

                try:
                    display(
                        Markdown(
                            "### 🤖 Відповідь БЕЗ RAG (тільки модель):"
                        )
                    )
                    no_rag_response = chatbot_without_rag_robotics(
                        query_text.value
                    )
                    display(Markdown(no_rag_response))

                    display(
                        Markdown(
                            "---\n### 📚 Відповідь З RAG (модель + база знань):"
                        )
                    )
                    rag_response, sources = chatbot_with_rag_robotics(
                        query_text.value
                    )
                    display(Markdown(rag_response))

                    if sources:
                        display(
                            Markdown(
                                f"\nВикористані джерела: {', '.join(sources)}"
                            )
                        )
                except Exception as e:
                    display(Markdown(f"❌ Помилка: {str(e)}"))

    button = widgets.Button(
        description="🚀 Отримати відповідь",
        button_style="success",
        layout=widgets.Layout(width="200px", height="40px"),
    )
    button.on_click(on_button_clicked)

    header = widgets.HTML(
        value="""
    <div style="background: linear-gradient(135deg, #0b1020 0%, #141b3a 100%); 
                padding: 20px; border-radius: 10px; margin-bottom: 20px;">
        <h1 style="color: #00e6b8; margin: 0;">
            🤖 Освітній RAG-асистент з робототехніки
        </h1>
        <p style="color: #a0a0a0; margin: 10px 0 0 0;">
            Асистент допоможе знайти відповідні курси або відповісти на запитання про робототехніку.
        </p>
    </div>
    """
    )

    info_panel = widgets.HTML(
        value="""
    <div style="background: #0d1117; padding: 15px; border-radius: 8px; 
                border-left: 4px solid #00e6b8; margin-bottom: 15px;">
        <b style="color: #58a6ff;">📚 Приклади доступних курсів:</b>
        <ul style="color: #c9d1d9; margin: 10px 0; font-size: 12px;">
            <li>Основи робототехніки</li>
            <li>Кінематика та динаміка роботів</li>
            <li>Актуатори та сенсори</li>
            <li>Системи керування роботами</li>
            <li>Мобільні роботи та SLAM</li>
            <li>Промислові роботи та маніпулятори</li>
            <li>Взаємодія людини та робота</li>
            <li>ШІ та сприйняття в робототехніці</li>
            <li>Ройова та багатороботна робототехніка</li>
            <li>Безпека та етика в робототехніці</li>
        </ul>
    </div>
    """
    )

    form_box = widgets.VBox(
        [action_dropdown, course_dropdown, query_text, button],
        layout=widgets.Layout(padding="10px"),
    )

    display(header)
    display(info_panel)
    display(form_box)
    display(output)